# Service predictors: payment, water state and waterpoint type

This focused notebook completes the initial **low-cardinality categorical** review of
`payment_type`, `water_quality`, `quality_group`, `quantity`, `waterpoint_type`, `waterpoint_type_group`. It describes the supplied training and test predictors,
then uses the labelled training rows to identify relationships worth
carrying into a leakage-safe modelling pipeline.

This notebook audits the retained canonical quantity/payment fields and two related categorical pairs.

This is exploratory evidence, not fitted preprocessing. Category pooling,
imputation, encoding and scaling must be learned inside each training fold.


## Consistent audit contract

Every focused predictor audit answers the same questions before adding
type-specific checks:

1. What is explicitly missing, and what looks like a sentinel?
2. What range or category coverage is present in training and test?
3. How much of the test set is exposed to unseen training levels?
4. Does the labelled distribution vary enough to justify retaining the field?
5. What exact baseline treatment follows from the evidence?

Target-rate tables flag support rather than treating tiny groups as reliable.
Train/test comparisons are descriptive and do not use the hidden test labels.


In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

stage_directory = next(
    candidate for candidate in [Path.cwd(), *Path.cwd().parents]
    if (candidate / "data" / "TrainingSetValues.csv").is_file()
)
source_directory = str((stage_directory / "src").resolve())
if source_directory not in sys.path:
    sys.path.insert(0, source_directory)

from predictor_audit import (
    MISSING_CATEGORY,
    analysis_categories,
    categorical_summary,
    categorical_target_profile,
    category_frequency_table,
    cramer_v,
    hierarchy_conflicts,
    hierarchy_summary,
    numeric_summary,
    numeric_target_summary,
    normalise_categories,
    sentinel_mask,
    source_blank_mask,
    text_normalisation_summary,
)
from source_data_validation import (
    validate_aligned_ids,
    validate_label_frame,
    validate_raw_feature_schema,
)

data_directory = stage_directory / "data"
training_features = pd.read_csv(
    data_directory / "TrainingSetValues.csv",
    keep_default_na=False,
)
training_labels = pd.read_csv(
    data_directory / "TrainingSetLabels.csv",
    keep_default_na=False,
)
test_features = pd.read_csv(
    data_directory / "TestSetValues.csv",
    keep_default_na=False,
)

validate_raw_feature_schema(training_features)
validate_raw_feature_schema(test_features)
validate_label_frame(training_labels)
validate_aligned_ids(training_features, training_labels)

training_data = training_features.merge(
    training_labels,
    on="id",
    validate="one_to_one",
)
audited_features = ['payment_type', 'water_quality', 'quality_group', 'quantity', 'waterpoint_type', 'waterpoint_type_group']
assert set(audited_features).issubset(training_features.columns)

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 80)
print(
    f"Validated {len(training_features):,} training rows and "
    f"{len(test_features):,} test rows for {len(audited_features)} predictors."
)


Validated 59,400 training rows and 14,850 test rows for 6 predictors.


## 1. Missingness, cardinality and test coverage

Source blanks, pandas nulls and configured sentinel strings are reported separately.
Semantic sentinels such as `unknown` stay visible in frequency and target tables;
they are not silently merged with blank values.
Rare means fewer than 50 training rows; it is a diagnostic threshold, not
a preprocessing choice. Total-variation distance compares marginal shares.


In [2]:
category_overview = categorical_summary(
    training_features,
    test_features,
    audited_features,
    rare_threshold=50,
    sentinel_tokens_by_column={},
)
display(category_overview)


,training explicit missing,training source blank rows,training sentinel rows,test explicit missing,test source blank rows,test sentinel rows,training levels,test levels,training levels with <50 rows,training rows in rare levels (%),test-only levels,test rows in unseen levels (%),training-only levels,marginal total-variation distance
feature,,,,,,,,,,,,,,
payment_type,0,0,8157,0,0,1992,7,7,0,0.00,0,0.0,0,0.0068
water_quality,0,0,1876,0,0,469,8,8,1,0.03,0,0.0,0,0.0016
quality_group,0,0,1876,0,0,469,6,6,0,0.00,0,0.0,0,0.0015
quantity,0,0,789,0,0,186,5,5,0,0.00,0,0.0,0,0.0035
waterpoint_type,0,0,0,0,0,0,7,7,1,0.01,0,0.0,0,0.0043
waterpoint_type_group,0,0,0,0,0,0,6,6,1,0.01,0,0.0,0,0.0043


## 2. Most common values


In [3]:
for feature in audited_features:
    print()
    print(feature)
    display(category_frequency_table(training_features, test_features, feature, top_n=10))



payment_type


,training rows,training (%),test rows,test (%)
payment_type,,,,
never pay,25348,42.67,6364,42.86
per bucket,8985,15.13,2281,15.36
monthly,8300,13.97,2097,14.12
unknown,8157,13.73,1992,13.41
on failure,3914,6.59,928,6.25
annually,3642,6.13,928,6.25
other,1054,1.77,260,1.75



water_quality


,training rows,training (%),test rows,test (%)
water_quality,,,,
soft,50818,85.55,12687,85.43
salty,4856,8.18,1226,8.26
unknown,1876,3.16,469,3.16
milky,804,1.35,201,1.35
coloured,490,0.82,133,0.9
salty abandoned,339,0.57,84,0.57
fluoride,200,0.34,44,0.3
fluoride abandoned,17,0.03,6,0.04



quality_group


...

,training rows,training (%),test rows,test (%)
quantity,,,,
enough,33186,55.87,8336,56.13
insufficient,15129,25.47,3767,25.37
dry,6246,10.52,1536,10.34
seasonal,4050,6.82,1025,6.9
unknown,789,1.33,186,1.25



waterpoint_type


,training rows,training (%),test rows,test (%)
waterpoint_type,,,,
communal standpipe,28522,48.02,7106,47.85
hand pump,17488,29.44,4396,29.6
other,6380,10.74,1630,10.98
communal standpipe multiple,6103,10.27,1508,10.15
improved spring,784,1.32,175,1.18
cattle trough,116,0.2,34,0.23
dam,7,0.01,1,0.01



waterpoint_type_group


,training rows,training (%),test rows,test (%)
waterpoint_type_group,,,,
communal standpipe,34625,58.29,8614,58.01
hand pump,17488,29.44,4396,29.6
other,6380,10.74,1630,10.98
improved spring,784,1.32,175,1.18
cattle trough,116,0.2,34,0.23
dam,7,0.01,1,0.01


## 3. Relationship with `status_group`

The tables display the most supported levels first and mark whether each
level has at least 150 training rows. Small groups are leads
for later validation, not stable target encodings.


In [4]:
for feature in audited_features:
    print()
    print(feature)
    profile = categorical_target_profile(
        training_data,
        feature,
        minimum_support=150,
    )
    display(profile.head(15))



payment_type


status_group,rows,meets support threshold,functional (%),functional needs repair (%),non functional (%)
payment_type,,,,,
never pay,25348,True,44.89,7.52,47.59
per bucket,8985,True,67.78,4.55,27.67
monthly,8300,True,66.05,11.17,22.78
unknown,8157,True,43.25,5.30,51.45
on failure,3914,True,62.06,7.08,30.86
annually,3642,True,75.23,6.78,17.98
other,1054,True,57.97,11.20,30.83



water_quality


status_group,rows,meets support threshold,functional (%),functional needs repair (%),non functional (%)
water_quality,,,,,
soft,50818,True,56.59,7.68,35.72
salty,4856,True,45.72,4.63,49.65
unknown,1876,True,14.07,1.87,84.06
milky,804,True,54.48,1.74,43.78
coloured,490,True,50.20,11.02,38.78
salty abandoned,339,True,51.33,21.24,27.43
fluoride,200,True,75.50,6.50,18.00
fluoride abandoned,17,False,35.29,0.00,64.71



quality_group


...

status_group,rows,meets support threshold,functional (%),functional needs repair (%),non functional (%)
quantity,,,,,
enough,33186,True,65.23,7.23,27.54
insufficient,15129,True,52.32,9.58,38.09
dry,6246,True,2.51,0.59,96.89
seasonal,4050,True,57.41,10.27,32.32
unknown,789,True,27.00,1.77,71.23



waterpoint_type


status_group,rows,meets support threshold,functional (%),functional needs repair (%),non functional (%)
waterpoint_type,,,,,
communal standpipe,28522,True,62.15,7.92,29.93
hand pump,17488,True,61.79,5.88,32.33
other,6380,True,13.17,4.59,82.24
communal standpipe multiple,6103,True,36.62,10.62,52.76
improved spring,784,True,71.81,10.84,17.35
cattle trough,116,False,72.41,1.72,25.86
dam,7,False,85.71,0.00,14.29



waterpoint_type_group


status_group,rows,meets support threshold,functional (%),functional needs repair (%),non functional (%)
waterpoint_type_group,,,,,
communal standpipe,34625,True,57.65,8.40,33.95
hand pump,17488,True,61.79,5.88,32.33
other,6380,True,13.17,4.59,82.24
improved spring,784,True,71.81,10.84,17.35
cattle trough,116,False,72.41,1.72,25.86
dam,7,False,85.71,0.00,14.29


## 4. Related-field consistency

A deterministic child-to-parent mapping makes the parent derivable from
the child in this dataset. That is redundancy evidence, not automatic
permission to discard the child: granularity, unseen levels and model
behaviour still determine which representation is safer.


In [5]:
hierarchy_relationships = [('water_quality', 'quality_group'), ('waterpoint_type', 'waterpoint_type_group')]
display(
    hierarchy_summary(
        training_features,
        test_features,
        hierarchy_relationships,
    )
)
for child, parent in hierarchy_relationships:
    conflicts = hierarchy_conflicts(training_features, child, parent)
    if not conflicts.empty:
        print()
        print(f"Training conflicts for {child} -> {parent}")
        display(conflicts)


complete rows  \
relationship                             frame                     
water_quality -> quality_group           training          59400   
                                         test              14850   
waterpoint_type -> waterpoint_type_group training          59400   
                                         test              14850   

                                                   child levels  \
relationship                             frame                    
water_quality -> quality_group           training             8   
                                         test                 8   
waterpoint_type -> waterpoint_type_group training             7   
                                         test                 7   

                                                   parent levels  \
relationship                             frame                     
water_quality -> quality_group           training              6   
                                         test                  6   
waterpoint_type -> waterpoint_type_group training              6   
                                         test                  6   

                                                   ambiguous child levels  \
relationship                             frame                              
water_quality -> quality_group           training                       0   
                                         test                           0   
waterpoint_type -> waterpoint_type_group training                       0   
                                         test                           0   

                                                   rows in ambiguous child levels  \
relationship                             frame                                      
water_quality -> quality_group           training                               0   
                                         test                                   0   
waterpoint_type -> waterpoint_type_group training                               0   
                                         test                                   0   

                                                   deterministic child-to-parent  \
relationship                             frame                                     
water_quality -> quality_group           training                           True   
                                         test                               True   
waterpoint_type -> waterpoint_type_group training                           True   
                                         test                               True   

                                                   one-to-one level mapping  
relationship                             frame                               
water_quality -> quality_group           training                     False  
                                         test                         False  
waterpoint_type -> waterpoint_type_group training                     False  
                                         test                         False

## Training/test handoff


In [6]:
display(
    category_overview[[
        "training levels",
        "test levels",
        "test-only levels",
        "test rows in unseen levels (%)",
        "training rows in rare levels (%)",
        "marginal total-variation distance",
    ]].sort_values("test rows in unseen levels (%)", ascending=False)
)


,training levels,test levels,test-only levels,test rows in unseen levels (%),training rows in rare levels (%),marginal total-variation distance
feature,,,,,,
payment_type,7,7,0,0.0,0.00,0.0068
water_quality,8,8,0,0.0,0.03,0.0016
quality_group,6,6,0,0.0,0.00,0.0015
quantity,5,5,0,0.0,0.00,0.0035
waterpoint_type,7,7,0,0.0,0.01,0.0043
waterpoint_type_group,6,6,0,0.0,0.01,0.0043


## Decision register

The register separates observed evidence from the proposed baseline action.
A retained field is still a candidate: later validation must show whether it
improves generalisation and whether a coarser related representation is safer.


In [7]:
decision_register = pd.DataFrame([{'feature': 'payment_type', 'quality finding': 'Seven test-covered levels; unknown is 51.45% non-functional versus 38.42% overall.', 'baseline treatment': 'Retain unknown/other explicitly after removing duplicate payment.', 'risk to verify': 'Payment patterns may proxy local administration.'}, {'feature': 'water_quality', 'quality finding': 'Eight levels map deterministically upward; unknown is 84.06% non-functional.', 'baseline treatment': 'Keep granular quality and rare-pool the 17-row fluoride-abandoned level.', 'risk to verify': 'Sentinel signal may reflect collection quality.'}, {'feature': 'quality_group', 'quality finding': 'Deterministic parent hides the salty versus salty-abandoned repair difference.', 'baseline treatment': 'Treat as likely redundant beside water_quality.', 'risk to verify': 'Coarsening can remove useful detail.'}, {'feature': 'quantity', 'quality finding': 'Dry covers 6,246 rows and is 96.89% non-functional; quantity_group is its duplicate.', 'baseline treatment': 'Retain as nominal; preserve unknown and never reintroduce quantity_group.', 'risk to verify': 'Strong association is legitimate but not causal evidence.'}, {'feature': 'waterpoint_type', 'quality finding': 'Seven levels; `other` is 82.24% non-functional and `dam` has only seven rows.', 'baseline treatment': 'Keep granular type with infrequent handling.', 'risk to verify': 'The dam estimate is too sparse to interpret.'}, {'feature': 'waterpoint_type_group', 'quality finding': 'Deterministic parent hides single-versus-multiple standpipe differences.', 'baseline treatment': 'Treat as likely redundant; use only as coarse ablation.', 'risk to verify': 'Coarsening can remove useful detail.'}])
display(decision_register.set_index("feature"))


,quality finding,baseline treatment,risk to verify
feature,,,
payment_type,Seven test-covered levels; unknown is 51.45% non-functional versus 38.42% ov...,Retain unknown/other explicitly after removing duplicate payment.,Payment patterns may proxy local administration.
water_quality,Eight levels map deterministically upward; unknown is 84.06% non-functional.,Keep granular quality and rare-pool the 17-row fluoride-abandoned level.,Sentinel signal may reflect collection quality.
quality_group,Deterministic parent hides the salty versus salty-abandoned repair difference.,Treat as likely redundant beside water_quality.,Coarsening can remove useful detail.
quantity,"Dry covers 6,246 rows and is 96.89% non-functional; quantity_group is its du...",Retain as nominal; preserve unknown and never reintroduce quantity_group.,Strong association is legitimate but not causal evidence.
waterpoint_type,Seven levels; `other` is 82.24% non-functional and `dam` has only seven rows.,Keep granular type with infrequent handling.,The dam estimate is too sparse to interpret.
waterpoint_type_group,Deterministic parent hides single-versus-multiple standpipe differences.,Treat as likely redundant; use only as coarse ablation.,Coarsening can remove useful detail.


### Handoff to modelling

Keep `payment_type` and `quantity`; the duplicate `payment` and `quantity_group` columns remain excluded by the settled structural policy.

- Preserve raw source frames and implement the stated sentinel rules on copies.
- Fit imputers, rare-level grouping and encoders on each training fold only.
- Map unseen validation or test categories to an explicit fallback.
- Compare the stated baseline treatment with a simple omission ablation.
- Revisit target-rate observations after the reproducible stratified split exists.
